In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from aquarel import load_theme
import numpy as np
import matplotlib as mpl
import matplotlib.patches as patches
from scipy.stats import ks_2samp
import itertools

theme = load_theme("minimal_light")
theme.apply()
theme.apply_transforms()
mpl.rcParams['axes.spines.left'] = True
mpl.rcParams['axes.spines.right'] = False
mpl.rcParams['axes.spines.top'] = False
mpl.rcParams['axes.spines.bottom'] = True
mpl.rcParams['axes.edgecolor'] = 'grey'

In [ ]:
FIGURE_DATA = Path("../figure_data")

ES_INPUT_MAP = {'es0': 3.0, 'es1': 2.0, 'es2': 1.0, 'es3': 0.75, 'es4': 0.5, 'es5': 0.25}

pt_concat = pd.read_csv(FIGURE_DATA / 'paramtraversal.csv')
pt_concat = pt_concat[pt_concat['Threshold'].isin([0.01])]
pt_concat = pt_concat[pt_concat['Bonferroni'].isin(['fdr_by'])]

print(pt_concat.shape)
print('Methods:', sorted(pt_concat['Method'].unique()))
print('Statistics:', sorted(pt_concat['Statistic'].unique()))
print('Effect sizes:', sorted(pt_concat['effect_size'].unique()))
pt_concat.head()

In [ ]:
def _rename_method(orig):
    """Strip 'simulate_glrates_' prefix for display labels."""
    prefix = 'simulate_glrates_'
    return orig[len(prefix):] if orig.startswith(prefix) else orig


def plot_trav(pt_concat, metric, es_input, ascending=True, highlight_bit_log_odds=True):
    """
    Heatmap of a single metric over the (Method × Statistic) grid at a given
    es_input value (float, e.g. 1.0 for es2).
    """
    df_filtered = pt_concat[pt_concat['es_input'] == es_input]
    piv = pd.pivot_table(df_filtered, index='Method', columns='Statistic', values=metric)

    original_index   = list(piv.index)
    original_columns = list(piv.columns)

    piv.index   = [_rename_method(i) for i in original_index]
    piv.columns = [i[1:-10] if i.startswith('_') else i[:-10] for i in original_columns]

    plt.figure(figsize=(4.5, 4.5))
    ax = sns.heatmap(piv,
                     cmap='Blues' if ascending else 'Blues_r',
                     annot=True, fmt=".3f",
                     cbar=False, cbar_kws={'ticks': []},
                     square=True)

    if highlight_bit_log_odds:
        try:
            y_pos = piv.index.get_loc('bit')
            x_pos = piv.columns.get_loc('log_odds_ratio')
            ax.add_patch(patches.Rectangle(
                (x_pos, y_pos), 1, 1,
                linewidth=3, edgecolor='red', facecolor='none', clip_on=False))
        except KeyError as e:
            print(f"Warning: highlight target not found: {e}")

    plt.ylabel('Simulation Method')
    plt.xlabel('Scoring Function')
    plt.title(metric.replace('_', ' '))
    plt.tight_layout()
    plt.show()


def test_metric(stat_name, value, es_input=1.0):
    """KS test comparing simulation methods on a given metric at es_input."""
    df_sub = pt_concat[
        (pt_concat["Statistic"] == stat_name) &
        (pt_concat["es_input"] == es_input)
    ].copy()
    all_ks = []
    for m1, m2 in itertools.combinations(df_sub["Method"].unique(), 2):
        stat, pval = ks_2samp(
            df_sub.loc[df_sub["Method"] == m1, value],
            df_sub.loc[df_sub["Method"] == m2, value],
        )
        all_ks.append({"Method1": m1, "Method2": m2, "KS_stat": stat, "pval": pval})
    return pd.DataFrame(all_ks)


def plot_ks(ks_df, metric_name, stat_col="KS_stat", pval_col="pval"):
    """Pairwise KS-statistic heatmap with significance stars."""
    methods = sorted(set(ks_df["Method1"]).union(ks_df["Method2"]))
    n = len(methods)

    matrix_stat = pd.DataFrame(np.nan, index=methods, columns=methods)
    matrix_pval = pd.DataFrame(np.nan, index=methods, columns=methods)

    for _, row in ks_df.iterrows():
        m1, m2 = row["Method1"], row["Method2"]
        matrix_stat.loc[m1, m2] = matrix_stat.loc[m2, m1] = row[stat_col]
        matrix_pval.loc[m1, m2] = matrix_pval.loc[m2, m1] = row[pval_col]

    def p_to_stars(p):
        if pd.isnull(p): return ""
        if p < 0.001: return "***"
        if p < 0.01:  return "**"
        if p < 0.05:  return "*"
        return "ns"

    star_annot = matrix_pval.map(p_to_stars)
    mask = np.eye(n, dtype=bool)

    plt.figure(figsize=(4.5, 4.5))
    ax = sns.heatmap(matrix_stat.astype(float),
                     cmap=sns.color_palette("viridis", as_cmap=True),
                     annot=star_annot, fmt="",
                     square=True,
                     cbar_kws={"label": "KS statistic", 'shrink': 0.5},
                     linewidths=0, vmin=0, vmax=1,
                     mask=mask)

    for i in range(n):
        ax.add_patch(plt.Rectangle((i, i), 1, 1, fill=True, color="lightgrey", lw=0))

    plt.title(metric_name.replace('_', ' '))
    plt.xlabel("")
    plt.ylabel("")
    plt.tight_layout()
    plt.show()

In [ ]:
### Figure S2 (1/4)
plot_trav(pt_concat=pt_concat, metric="PR_AUC_Positive", es_input=1.0)

In [ ]:
### Figure S2 (2/4)
plot_trav(pt_concat=pt_concat, metric="FPR_Positive", es_input=1.0, ascending=False)

In [ ]:
### Figure S2 (3/4)
plot_trav(pt_concat=pt_concat, metric="PR_AUC_Negative", es_input=1.0)

In [ ]:
### Figure S2 (4/4)
plot_trav(pt_concat=pt_concat, metric="FPR_Negative", es_input=1.0, ascending=False)

In [ ]:
stat_name = "_log_odds_ratio_statistic"
value = 'FPR_Negative'
ks = test_metric(stat_name, value, es_input=1.0)
plot_ks(ks, value)

In [ ]:
plot_trav(pt_concat, "Precision_Positive",  1.0)
plot_trav(pt_concat, "Recall_Positive",     1.0)
plot_trav(pt_concat, "Precision_Negative",  1.0)
plot_trav(pt_concat, "Recall_Negative",     1.0)
plot_trav(pt_concat, "F1_Positive",         1.0)
plot_trav(pt_concat, "F1_Negative",         1.0)
plot_trav(pt_concat, "AUC_ROC_Positive",    1.0)
plot_trav(pt_concat, "AUC_ROC_Negative",    1.0)

In [ ]:
def plot_trav_and_ks(pt_concat, metric, es_input, stat_name="_log_odds_ratio_statistic", ascending=True):
    """Side-by-side: traversal heatmap (left) + pairwise KS comparison (right)."""

    # ---- LEFT: traversal pivot ----
    piv = pd.pivot_table(
        pt_concat[pt_concat['es_input'] == es_input],
        index='Method', columns='Statistic', values=metric,
    )
    piv.index   = [_rename_method(i) for i in piv.index]
    piv.columns = [i[1:-10] if i.startswith('_') else i[:-10] for i in piv.columns]

    method_order   = list(piv.index)
    n              = len(method_order)
    stat_name_col  = stat_name[1:-10] if stat_name.startswith('_') else stat_name[:-10]

    # ---- Compute KS statistics (same ordering as traversal pivot) ----
    df_stat = pt_concat[
        (pt_concat["Statistic"] == stat_name) &
        (pt_concat["es_input"]  == es_input)
    ]

    matrix_stat = pd.DataFrame(np.nan, index=method_order, columns=method_order)
    matrix_pval = pd.DataFrame(np.nan, index=method_order, columns=method_order)

    for m1, m2 in itertools.combinations(sorted(df_stat["Method"].unique()), 2):
        stat, pval = ks_2samp(
            df_stat.loc[df_stat["Method"] == m1, metric],
            df_stat.loc[df_stat["Method"] == m2, metric],
        )
        lm1, lm2 = _rename_method(m1), _rename_method(m2)
        if lm1 in matrix_stat.index and lm2 in matrix_stat.columns:
            matrix_stat.loc[lm1, lm2] = matrix_stat.loc[lm2, lm1] = stat
            matrix_pval.loc[lm1, lm2] = matrix_pval.loc[lm2, lm1] = pval

    mask = np.eye(n, dtype=bool)

    def p_to_stars(p):
        if pd.isnull(p): return ""
        if p < 0.001: return "***"
        if p < 0.01:  return "**"
        if p < 0.05:  return "*"
        return "ns"

    annot_stars = matrix_pval.map(p_to_stars)

    # ---- Figure ----
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.5),
                             gridspec_kw={"width_ratios": [1.35, 1]})

    ax0 = axes[0]
    sns.heatmap(piv, cmap='Blues' if ascending else 'Blues_r',
                annot=True, fmt=".3f", cbar=False, square=True, ax=ax0)
    ax0.set_ylabel("Simulation Method")
    ax0.set_xlabel("Scoring Function")
    ax0.set_title(metric.replace('_', ' '))
    ax0.set_xticklabels(ax0.get_xticklabels(), rotation=45, ha='right')

    if stat_name_col in list(piv.columns):
        col_idx = list(piv.columns).index(stat_name_col)
        ax0.add_patch(plt.Rectangle((col_idx, 0), 1, piv.shape[0],
                                    fill=False, lw=3, edgecolor='red', clip_on=False))

    ax1 = axes[1]
    sns.heatmap(matrix_stat.astype(float),
                cmap=sns.color_palette("viridis", as_cmap=True),
                annot=annot_stars, fmt="",
                square=True, ax=ax1,
                cbar_kws={"label": "KS statistic", "shrink": 0.6},
                vmin=0, vmax=1, mask=mask, linewidths=0)

    for i in range(n):
        ax1.add_patch(plt.Rectangle((i, i), 1, 1, color="lightgrey", fill=True, lw=0))

    ax1.set_title(f"Statistical Comparison: {stat_name.replace('_', ' ')}")
    ax1.set_ylabel("")
    ax1.set_yticks([])
    ax1.set_xticklabels(ax1.get_xticklabels(), rotation=0)

    plt.tight_layout()
    return fig


es_input = 1.0   # es2
x = plot_trav_and_ks(pt_concat=pt_concat, metric="PR_AUC_Positive",  es_input=es_input)
x = plot_trav_and_ks(pt_concat=pt_concat, metric="PR_AUC_Negative",  es_input=es_input)
x = plot_trav_and_ks(pt_concat=pt_concat, metric="FPR_Positive",     es_input=es_input, ascending=False)
x = plot_trav_and_ks(pt_concat=pt_concat, metric="FPR_Negative",     es_input=es_input, ascending=False)